# 07 - NLTK: Natural Language Toolkit — Introduction & Basic Operations

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what NLTK is and how it differs from `src/utils/text_preprocessing.py`.
- Download and manage NLTK corpora and trained models.
- Use NLTK's **linguistically-correct tokenizers**: Treebank word, Punkt sentence, TweetTokenizer.
- Filter text with NLTK **stopword lists** across 25 languages.
- Apply **Porter**, **Snowball**, and **Lancaster** stemmers and compare aggressiveness.
- Use **WordNetLemmatizer** with POS tags for accurate lemmatization.
- Run **POS tagging** with the Penn Treebank tagset.
- Identify named entities with **`ne_chunk`**.
- Query **WordNet** for synonyms, antonyms, hypernyms, and semantic similarity.
- Compute **frequency distributions** and **bigram collocations**.
- Choose when to use NLTK vs the lightweight custom utilities.

## Prerequisites

- Completed Notebook 01 (Text Preprocessing & Tokenization).
- Completed Notebook 03 (Stemming, Lemmatization, Stop Words).
- `pip install nltk` — run once before this notebook.

## Table of Contents

1. [What is NLTK?](#1-what-is-nltk)
2. [Installation and Data Downloads](#2-installation-and-data-downloads)
3. [Tokenization](#3-tokenization)
4. [Stopwords](#4-stopwords)
5. [Stemming](#5-stemming)
6. [Lemmatization with WordNet](#6-lemmatization-with-wordnet)
7. [Part-of-Speech Tagging](#7-part-of-speech-tagging)
8. [Named Entity Recognition](#8-named-entity-recognition)
9. [WordNet Semantic Database](#9-wordnet-semantic-database)
10. [Frequency Distributions and Collocations](#10-frequency-distributions-and-collocations)
11. [NLTK vs Custom text_preprocessing.py](#11-nltk-vs-custom-text_preprocessingpy)
12. [Building an NLTK Pipeline](#12-building-an-nltk-pipeline)
13. [Common Mistakes and Debugging Tips](#13-common-mistakes-and-debugging-tips)
14. [Exercises](#14-exercises)
15. [Exercise Solutions](#15-exercise-solutions)

In [ ]:
# ---- Environment setup ----
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import re
import string
from collections import Counter
from typing import List, Dict, Tuple, Optional

print("Setup complete.")

## 1. What is NLTK?

**NLTK (Natural Language Toolkit)** is the most widely-used Python library for classical NLP research and education. Created in 2001 at the University of Pennsylvania, it covers every layer of the NLP stack:

| Layer | What NLTK provides |
|-------|--------------------|
| **Tokenization** | Linguistically-trained word and sentence splitters |
| **Morphology** | Porter / Snowball / Lancaster stemmers, WordNet lemmatizer |
| **Syntax** | POS tagger, regex chunker, dependency grammar |
| **Semantics** | WordNet: synonyms, antonyms, hypernyms, similarity |
| **Corpora** | 50+ bundled datasets (Gutenberg, Reuters, Brown, Twitter, ...) |
| **Statistics** | FreqDist, bigram/trigram collocations, PMI |

### How NLTK differs from `src/utils/text_preprocessing.py`

| Aspect | `text_preprocessing.py` | NLTK |
|--------|:------------------------:|:----:|
| **Purpose** | Produces integer sequences for DL models | General-purpose NLP research |
| **Tokenization** | Regex-based (fast, naive) | Trained models (accurate, slower) |
| **Vocab builder** | YES — word→index for embedding layers | No |
| **Stemming / Lemmatization** | No | Full support |
| **POS / NER** | No | Full support |
| **Semantic knowledge** | No | WordNet (155 k words) |
| **Dependencies** | stdlib only | `nltk` + data downloads |
| **Speed** | Very fast | Moderate |

> **Rule of thumb:** Use NLTK for linguistically accurate cleaning; use `text_preprocessing.py` to convert the result into tensors for training.

## 2. Installation and Data Downloads

NLTK separates the **library** (code) from the **data** (corpora, trained models). Install them independently:

```bash
pip install nltk
```

In [1]:
import nltk

# Packages used in this notebook
packages = [
    "punkt",                          # sentence tokenizer model
    "punkt_tab",                      # punkt lookup tables (newer NLTK)
    "averaged_perceptron_tagger",     # POS tagger
    "averaged_perceptron_tagger_eng", # English POS tagger tables
    "maxent_ne_chunker",              # named entity recognizer
    "maxent_ne_chunker_tab",          # NER lookup tables
    "words",                          # English word list
    "stopwords",                      # stopword lists (25 languages)
    "wordnet",                        # WordNet lexical database
    "omw-1.4",                        # Open Multilingual Wordnet
    "brown",                          # Brown corpus
    "gutenberg",                      # Project Gutenberg texts
]

for pkg in packages:
    result = nltk.download(pkg, quiet=True)
    status = "downloaded" if result else "already present"
    print(f"  {pkg:<44} {status}")

print(f"\nNLTK version: {nltk.__version__}")
print("All NLTK data ready.")

  punkt                                        downloaded
  punkt_tab                                    downloaded
  averaged_perceptron_tagger                   downloaded
  averaged_perceptron_tagger_eng               downloaded
  maxent_ne_chunker                            downloaded
  maxent_ne_chunker_tab                        downloaded
  words                                        downloaded
  stopwords                                    downloaded
  wordnet                                      downloaded
  omw-1.4                                      downloaded
  brown                                        downloaded
  gutenberg                                    downloaded

NLTK version: 3.9.2
All NLTK data ready.


## 3. Tokenization

NLTK provides several tokenizers, each trained on different corpora and optimized for different domains.

### 3.1 Word Tokenization — Treebank

NLTK's default `word_tokenize` uses **Penn Treebank** rules. Key behaviors vs a simple regex split:

| Text | Regex split | `word_tokenize` |
|------|-------------|------------------|
| `don't` | `["don", "'", "t"]` | `["do", "n't"]` |
| `Dr.` | `["Dr", "."]` | `["Dr."]` |
| `$9.99` | `["$", "9", ".", "99"]` | `["$", "9.99"]` |

In [6]:
import sys
import re
sys.path.insert(0, "../..")

In [4]:
from nltk.tokenize import word_tokenize
from src.utils.text_preprocessing import basic_tokenize

samples = [
    "I don't think that's right.",
    "Dr. Smith visited the U.S.A. for $9.99 per night.",
    "She said, \"Hello, world!\" and waved.",
    "They've been there since 2001-09-11.",
]

print(f"{'Input':<50} {'basic_tokenize':<45} {'word_tokenize'}")
print("-" * 140)
for text in samples:
    basic = str(basic_tokenize(text))
    nltk_t = str(word_tokenize(text))
    print(f"{text:<50} {basic:<45} {nltk_t}")

print()
print("Key difference: word_tokenize handles contractions and abbreviations correctly.")

Input                                              basic_tokenize                                word_tokenize
--------------------------------------------------------------------------------------------------------------------------------------------
I don't think that's right.                        ['i', 'don', "'", 't', 'think', 'that', "'", 's', 'right', '.'] ['I', 'do', "n't", 'think', 'that', "'s", 'right', '.']
Dr. Smith visited the U.S.A. for $9.99 per night.  ['dr', '.', 'smith', 'visited', 'the', 'u', '.', 's', '.', 'a', '.', 'for', '$', '9', '.', '99', 'per', 'night', '.'] ['Dr.', 'Smith', 'visited', 'the', 'U.S.A.', 'for', '$', '9.99', 'per', 'night', '.']
She said, "Hello, world!" and waved.               ['she', 'said', ',', '"', 'hello', ',', 'world', '!', '"', 'and', 'waved', '.'] ['She', 'said', ',', '``', 'Hello', ',', 'world', '!', "''", 'and', 'waved', '.']
They've been there since 2001-09-11.               ['they', "'", 've', 'been', 'there', 'since', '2001', '-',

### 3.2 Sentence Tokenization — Punkt

`sent_tokenize` uses the **Punkt** unsupervised algorithm, which learns abbreviations vs sentence boundaries from training data. It correctly handles `Dr.`, `p.m.`, `U.S.A.` without false splits.

In [7]:
from nltk.tokenize import sent_tokenize

tricky = (
    "Dr. Smith went to Washington D.C. at 3 p.m. "
    "He arrived safely. The meeting lasted until 5 p.m. "
    "Mr. Jones asked: will there be a follow-up? "
    "Yes, next week on Jan. 15."
)

# Naive regex split on period + space
naive = re.split(r'\.\s+', tricky)
punkt = sent_tokenize(tricky)

print("Naive regex split:")
for i, s in enumerate(naive, 1):
    print(f"  [{i}] {s}")

print()
print("Punkt sent_tokenize:")
for i, s in enumerate(punkt, 1):
    print(f"  [{i}] {s}")

print()
print("Punkt correctly treats Dr., p.m., D.C., Mr., Jan. as abbreviations — no false splits.")

Naive regex split:
  [1] Dr
  [2] Smith went to Washington D.C
  [3] at 3 p.m
  [4] He arrived safely
  [5] The meeting lasted until 5 p.m
  [6] Mr
  [7] Jones asked: will there be a follow-up? Yes, next week on Jan
  [8] 15.

Punkt sent_tokenize:
  [1] Dr. Smith went to Washington D.C. at 3 p.m.
  [2] He arrived safely.
  [3] The meeting lasted until 5 p.m. Mr. Jones asked: will there be a follow-up?
  [4] Yes, next week on Jan. 15.

Punkt correctly treats Dr., p.m., D.C., Mr., Jan. as abbreviations — no false splits.


### 3.3 TweetTokenizer

Social media text contains @mentions, #hashtags, emoticons, repeated characters (`loooove`), and URLs. `TweetTokenizer` handles all of these without mangling them.

In [8]:
from nltk.tokenize import TweetTokenizer

default_tok = TweetTokenizer()
norm_tok    = TweetTokenizer(strip_handles=True, reduce_len=True, preserve_case=False)

tweets = [
    "@UserName I loooooove this product!! #amazing #NLP :) <3",
    "RT @CNN: Breaking news!!! Check https://t.co/abc123 #news",
    "Can't wait for the weekend!!! \U0001F60A #TGIF @bestfriend",
]

print("TweetTokenizer — default vs normalized:")
print("=" * 65)
for tweet in tweets:
    print(f"  Input:      {tweet}")
    print(f"  Default:    {default_tok.tokenize(tweet)}")
    print(f"  Normalized: {norm_tok.tokenize(tweet)}")
    print()

print("strip_handles removes @mentions, reduce_len caps repeated chars to 3.")

TweetTokenizer — default vs normalized:
  Input:      @UserName I loooooove this product!! #amazing #NLP :) <3
  Default:    ['@UserName', 'I', 'loooooove', 'this', 'product', '!', '!', '#amazing', '#NLP', ':)', '<3']
  Normalized: ['i', 'looove', 'this', 'product', '!', '!', '#amazing', '#nlp', ':)', '<3']

  Input:      RT @CNN: Breaking news!!! Check https://t.co/abc123 #news
  Default:    ['RT', '@CNN', ':', 'Breaking', 'news', '!', '!', '!', 'Check', 'https://t.co/abc123', '#news']
  Normalized: ['rt', ':', 'breaking', 'news', '!', '!', '!', 'check', 'https://t.co/abc123', '#news']

  Input:      Can't wait for the weekend!!! 😊 #TGIF @bestfriend
  Default:    ["Can't", 'wait', 'for', 'the', 'weekend', '!', '!', '!', '😊', '#TGIF', '@bestfriend']
  Normalized: ["can't", 'wait', 'for', 'the', 'weekend', '!', '!', '!', '😊', '#tgif']

strip_handles removes @mentions, reduce_len caps repeated chars to 3.


## 4. Stopwords

NLTK ships stopword lists for **25 languages**. Each list contains common function words that carry little semantic content for bag-of-words tasks.

In [9]:
from nltk.corpus import stopwords

langs = stopwords.fileids()
print(f"Languages available ({len(langs)}): {', '.join(langs)}")
print()

en_stops = set(stopwords.words("english"))
print(f"English stopwords ({len(en_stops)} total, first 30 alphabetically):")
print(sorted(en_stops)[:30])
print()

for lang in ["french", "german", "spanish"]:
    sw = stopwords.words(lang)
    print(f"{lang.capitalize():<10} ({len(sw)} words) sample: {sw[:8]}")

Languages available (33): albanian, arabic, azerbaijani, basque, belarusian, bengali, catalan, chinese, danish, dutch, english, finnish, french, german, greek, hebrew, hinglish, hungarian, indonesian, italian, kazakh, nepali, norwegian, portuguese, romanian, russian, slovene, spanish, swedish, tajik, tamil, turkish, uzbek

English stopwords (198 total, first 30 alphabetically):
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't"]

French     (157 words) sample: ['au', 'aux', 'avec', 'ce', 'ces', 'dans', 'de', 'des']
German     (232 words) sample: ['aber', 'alle', 'allem', 'allen', 'aller', 'alles', 'als', 'also']
Spanish    (313 words) sample: ['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los']


In [10]:
import nltk
nltk.download("indian")

from nltk.corpus import indian

print(indian.words("hindi.pos")[:20])
print(indian.tagged_words("hindi.pos")[:20])

['पूर्ण', 'प्रतिबंध', 'हटाओ', ':', 'इराक', 'संयुक्त', 'राष्ट्र', '।', 'इराक', 'के', 'विदेश', 'मंत्री', 'ने', 'अमरीका', 'के', 'उस', 'प्रस्ताव', 'का', 'मजाक', 'उड़ाया']
[('पूर्ण', 'JJ'), ('प्रतिबंध', 'NN'), ('हटाओ', 'VFM'), (':', 'SYM'), ('इराक', 'NNP'), ('संयुक्त', 'NNC'), ('राष्ट्र', 'NN'), ('।', 'SYM'), ('इराक', 'NNP'), ('के', 'PREP'), ('विदेश', 'NNC'), ('मंत्री', 'NN'), ('ने', 'PREP'), ('अमरीका', 'NNP'), ('के', 'PREP'), ('उस', 'PRP'), ('प्रस्ताव', 'NN'), ('का', 'PREP'), ('मजाक', 'NVB'), ('उड़ाया', 'VFM')]


[nltk_data] Downloading package indian to
[nltk_data]     C:\Users\shrip\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\indian.zip.


In [ ]:
def remove_stopwords(tokens: List[str], lang: str = "english") -> List[str]:
    stop_set = set(stopwords.words(lang))
    return [t for t in tokens if t.lower() not in stop_set]

texts = [
    "Deep learning models are trained on large datasets.",
    "This movie is not good at all.",          # DANGER: 'not' is a stopword!
    "No problem with the service whatsoever.",  # DANGER: 'no' is a stopword!
]

print("Stopword removal — including the negation trap:")
print("=" * 60)
for text in texts:
    tokens   = word_tokenize(text.lower())
    filtered = remove_stopwords(tokens)
    removed  = [t for t in tokens if t not in filtered]
    print(f"  Input:   {text}")
    print(f"  Kept:    {filtered}")
    print(f"  Removed: {removed}")
    print()

negation_stops = {w for w in en_stops if w in {"no", "not", "nor", "neither", "never"}}
print(f"Negation words inside NLTK stoplist: {sorted(negation_stops)}")
print("Always use keep_negation=True for sentiment tasks!")

## 5. Stemming

NLTK offers three stemmers with different levels of aggressiveness:

| Stemmer | Algorithm | Aggressiveness |
|---------|-----------|:--------------:|
| **Porter** | 5-step suffix rules (1980) | Medium |
| **Snowball** (Porter2) | Improved Porter, 15 languages | Medium |
| **Lancaster** | Iterative suffix stripping | Very aggressive |

In [11]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter    = PorterStemmer()
snowball  = SnowballStemmer("english")
lancaster = LancasterStemmer()

test_words = [
    "running", "generously", "fairly", "university",
    "computational", "maximum", "presumably", "eating",
    "happiness", "connections",
]

print(f"{'Word':<18} {'Porter':<16} {'Snowball':<16} {'Lancaster'}")
print("-" * 65)
for w in test_words:
    print(f"{w:<18} {porter.stem(w):<16} {snowball.stem(w):<16} {lancaster.stem(w)}")

print()
print("Lancaster is the most aggressive — stems may not be valid words.")
print("Porter and Snowball are more conservative and often preferred.")

Word               Porter           Snowball         Lancaster
-----------------------------------------------------------------
running            run              run              run
generously         gener            generous         gen
fairly             fairli           fair             fair
university         univers          univers          univers
computational      comput           comput           comput
maximum            maximum          maximum          maxim
presumably         presum           presum           presum
eating             eat              eat              eat
happiness          happi            happi            happy
connections        connect          connect          connect

Lancaster is the most aggressive — stems may not be valid words.
Porter and Snowball are more conservative and often preferred.


In [12]:
# Show vocabulary collapse — why stemming helps search/retrieval
word_groups = [
    ["connect", "connected", "connecting", "connection", "connections"],
    ["study", "studies", "studying", "studied"],
    ["university", "universities", "universal", "universe"],
]

print("Vocabulary collapse via Porter stemming:")
print("=" * 55)
for group in word_groups:
    stems = [porter.stem(w) for w in group]
    print(f"  Words:  {group}")
    print(f"  Stems:  {stems}  ->  unique: {set(stems)}")
    print()

# Snowball supports multiple languages
print(f"Snowball supported languages: {SnowballStemmer.languages}")

Vocabulary collapse via Porter stemming:
  Words:  ['connect', 'connected', 'connecting', 'connection', 'connections']
  Stems:  ['connect', 'connect', 'connect', 'connect', 'connect']  ->  unique: {'connect'}

  Words:  ['study', 'studies', 'studying', 'studied']
  Stems:  ['studi', 'studi', 'studi', 'studi']  ->  unique: {'studi'}

  Words:  ['university', 'universities', 'universal', 'universe']
  Stems:  ['univers', 'univers', 'univers', 'univers']  ->  unique: {'univers'}

Snowball supported languages: ('arabic', 'danish', 'dutch', 'english', 'finnish', 'french', 'german', 'hungarian', 'italian', 'norwegian', 'porter', 'portuguese', 'romanian', 'russian', 'spanish', 'swedish')


## 6. Lemmatization with WordNet

NLTK's `WordNetLemmatizer` reduces words to their **lemma** — a valid dictionary form — via WordNet lookup.

### 6.1 Basic Lemmatization

In [13]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

test_words = ["running", "cats", "mice", "geese", "better",
              "studies", "was", "children", "universities", "caring"]

print("WordNetLemmatizer (default: noun POS):")
print(f"{'Word':<18} {'Lemma (noun)'}")
print("-" * 35)
for w in test_words:
    print(f"{w:<18} {lemmatizer.lemmatize(w)}")

print()
print("'running' stays 'running' — wrong because the default POS is noun.")
print("Pass pos='v' for verbs (Section 6.2).")

WordNetLemmatizer (default: noun POS):
Word               Lemma (noun)
-----------------------------------
running            running
cats               cat
mice               mouse
geese              goose
better             better
studies            study
was                wa
children           child
universities       university
caring             caring

'running' stays 'running' — wrong because the default POS is noun.
Pass pos='v' for verbs (Section 6.2).


### 6.2 POS-Aware Lemmatization

The correct workflow is: **tokenize → POS-tag → map tag → lemmatize**.

In [15]:
from nltk.corpus import wordnet as wn
from nltk import pos_tag
from typing import List, Tuple

def treebank_to_wn(tag: str) -> str:
    """Map Penn Treebank tag to WordNet POS constant."""
    if tag.startswith("J"): return wn.ADJ
    if tag.startswith("V"): return wn.VERB
    if tag.startswith("N"): return wn.NOUN
    if tag.startswith("R"): return wn.ADV
    return wn.NOUN


def lemmatize_sentence(text: str) -> List[Tuple[str, str, str]]:
    """Return (original, POS_tag, lemma) for each token."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    return [
        (tok, tag, lemmatizer.lemmatize(tok.lower(), pos=treebank_to_wn(tag)))
        for tok, tag in tagged
    ]


# POS disambiguation
print("Same word, different POS -> different lemma:")
print(f"{'Word':<10} {'POS':<6} {'Lemma'}")
print("-" * 28)
cases = [("better","a"),("better","n"),("better","v"),
         ("running","v"),("running","n"),("leaves","v"),("leaves","n")]
for word, pos in cases:
    wn_pos = {"v":wn.VERB,"n":wn.NOUN,"a":wn.ADJ,"r":wn.ADV}.get(pos, wn.NOUN)
    print(f"{word:<10} {pos:<6} {lemmatizer.lemmatize(word, pos=wn_pos)}")

print()
# Full sentence demo
sent = "The geese were flying faster and the mice were running away."
triples = lemmatize_sentence(sent)
print(f"Sentence: {sent}")
print(f"\n{'Original':<15} {'Tag':<8} {'Lemma'}")
print("-" * 35)
for orig, tag, lemma in triples:
    changed = " <--" if orig.lower() != lemma else ""
    print(f"{orig:<15} {tag:<8} {lemma}{changed}")

Same word, different POS -> different lemma:
Word       POS    Lemma
----------------------------
better     a      good
better     n      better
better     v      better
running    v      run
running    n      running
leaves     v      leave
leaves     n      leaf

Sentence: The geese were flying faster and the mice were running away.

Original        Tag      Lemma
-----------------------------------
The             DT       the
geese           JJ       geese
were            VBD      be <--
flying          VBG      fly <--
faster          RBR      faster
and             CC       and
the             DT       the
mice            NN       mouse <--
were            VBD      be <--
running         VBG      run <--
away            RB       away
.               .        .


### 6.3 Stemming vs Lemmatization

In [ ]:
comparison = [
    ("running", "v"), ("better", "a"), ("mice", "n"), ("was", "v"),
    ("happiness", "n"), ("university", "n"), ("caring", "v"), ("geese", "n"),
]

print(f"{'Word':<14} {'Porter':<14} {'Snowball':<14} {'Lemma (POS)'}")
print("-" * 58)
for word, pos in comparison:
    wn_pos = {"v":wn.VERB,"n":wn.NOUN,"a":wn.ADJ,"r":wn.ADV}[pos]
    p = porter.stem(word)
    s = snowball.stem(word)
    l = lemmatizer.lemmatize(word, pos=wn_pos)
    print(f"{word:<14} {p:<14} {s:<14} {l}")

print()
print("Stems marked below are NOT valid English words:")
for w, _ in comparison:
    ps = porter.stem(w)
    from nltk.corpus import words as nltk_words
    valid = set(w.lower() for w in nltk_words.words())
    if ps not in valid:
        print(f"  * '{ps}'  (stem of '{w}')")

## 7. Part-of-Speech Tagging

NLTK's `pos_tag` uses a pre-trained **averaged perceptron** model and returns **Penn Treebank** tags.

### Common Penn Treebank Tags

| Tag | Meaning | Example |
|-----|---------|--------|
| `NN` / `NNS` | Noun singular / plural | dog / dogs |
| `NNP` / `NNPS` | Proper noun singular / plural | London / Alps |
| `VB` / `VBD` / `VBG` | Verb base / past / gerund | run / ran / running |
| `VBN` / `VBP` / `VBZ` | Past participle / non-3rd / 3rd person | run / run / runs |
| `JJ` / `JJR` / `JJS` | Adj / comparative / superlative | big / bigger / biggest |
| `RB` | Adverb | quickly |
| `DT` | Determiner | the, a |
| `IN` | Preposition | in, on, at |
| `PRP` | Personal pronoun | I, he, she |
| `CD` | Cardinal number | one, 3 |

In [ ]:
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Apple is planning to open new stores in New York.",
    "Natural language processing is not easy to learn.",
]

for sent in sentences:
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)
    print(f"Sentence: {sent}")
    print(f"Tagged:   {tagged}")
    print()

In [ ]:
# POS tagging disambiguates word sense
ambiguous = [
    "I saw the man with the telescope.",      # saw = VBD (verb, see)
    "The carpenter used a saw carefully.",    # saw = NN  (noun)
    "The leaves fell in autumn.",              # leaves = NNS (noun)
    "He leaves the house at 8 am.",            # leaves = VBZ (verb)
]

print("POS tagging disambiguates word sense:")
print("=" * 55)
for sent in ambiguous:
    tagged = pos_tag(word_tokenize(sent))
    for word, tag in tagged:
        if word.lower() in {"saw", "leaves"}:
            print(f"  '{word}' tagged as [{tag}] in: \"{sent}\"")
            break

## 8. Named Entity Recognition

NLTK's `ne_chunk` identifies named entities using a **MaxEnt classifier** trained on the ACE corpus.

| Label | Meaning | Examples |
|-------|---------|----------|
| `PERSON` | People | Barack Obama, Ada Lovelace |
| `ORGANIZATION` | Companies, agencies | Apple, NASA, WHO |
| `GPE` | Geo-political entity | New York, France |
| `LOCATION` | Non-GPE places | Mount Everest |
| `FACILITY` | Buildings | Golden Gate Bridge |

In [ ]:
from nltk import ne_chunk
from nltk.tree import Tree

def extract_entities(text: str) -> List[Tuple[str, str]]:
    """Return (entity_text, entity_type) pairs from a sentence."""
    tokens  = word_tokenize(text)
    tagged  = pos_tag(tokens)
    chunked = ne_chunk(tagged, binary=False)
    entities = []
    for subtree in chunked:
        if isinstance(subtree, Tree):
            text_span = " ".join(w for w, t in subtree.leaves())
            entities.append((text_span, subtree.label()))
    return entities


ner_texts = [
    "Barack Obama was born in Honolulu, Hawaii and attended Harvard University.",
    "Apple Inc. opened a new store in New York near the Empire State Building.",
    "NASA launched the James Webb Space Telescope in December 2021.",
    "Elon Musk founded SpaceX and Tesla Motors in California.",
]

print("Named Entity Recognition:")
print("=" * 65)
for text in ner_texts:
    entities = extract_entities(text)
    print(f"Text: {text}")
    for ent, etype in entities:
        print(f"  [{etype}]  '{ent}'")
    print()

In [ ]:
# Show the raw parse tree
text   = "Google CEO Sundar Pichai announced a new product in San Francisco."
tokens = word_tokenize(text)
tagged = pos_tag(tokens)
tree   = ne_chunk(tagged)

print(f"Input: {text}")
print()
print("NE Chunk Tree:")
print(tree)

## 9. WordNet Semantic Database

**WordNet** is a lexical database of English organized into **synsets** (synonym sets). Each synset represents one distinct concept and links to definitions, examples, and related concepts.

WordNet contains **155,000 words** in **175,000 synsets**.

### 9.1 Synsets and Definitions

In [ ]:
from nltk.corpus import wordnet as wn

# 'bank' is famously polysemous
synsets = wn.synsets("bank")
print(f"WordNet synsets for 'bank' ({len(synsets)} found):")
print("=" * 65)
for syn in synsets[:6]:   # show first 6
    print(f"  Name:       {syn.name()}")
    print(f"  POS:        {syn.pos()}")
    print(f"  Definition: {syn.definition()}")
    if syn.examples():
        print(f"  Example:    {syn.examples()[0]}")
    print(f"  Lemmas:     {[l.name() for l in syn.lemmas()]}")
    print()

### 9.2 Synonyms and Antonyms

In [ ]:
def get_synonyms(word: str, pos: Optional[str] = None) -> set:
    synonyms = set()
    for syn in wn.synsets(word, pos=pos):
        for lemma in syn.lemmas():
            if lemma.name().lower() != word.lower():
                synonyms.add(lemma.name().replace("_", " "))
    return synonyms


def get_antonyms(word: str, pos: Optional[str] = None) -> set:
    antonyms = set()
    for syn in wn.synsets(word, pos=pos):
        for lemma in syn.lemmas():
            for ant in lemma.antonyms():
                antonyms.add(ant.name().replace("_", " "))
    return antonyms


words_to_check = [
    ("happy", wn.ADJ),
    ("fast",  wn.ADJ),
    ("good",  wn.ADJ),
    ("run",   wn.VERB),
    ("big",   wn.ADJ),
]

print(f"{'Word':<10} {'Synonyms (top 5)':<50} {'Antonyms'}")
print("-" * 90)
for word, pos in words_to_check:
    syns = sorted(get_synonyms(word, pos=pos))[:5]
    ants = sorted(get_antonyms(word, pos=pos))[:3]
    print(f"{word:<10} {str(syns):<50} {str(ants)}")

### 9.3 Hypernyms and Hyponyms

WordNet organizes concepts in an **`is-a` hierarchy**:
- **Hypernym** — more general concept: `dog → canine → animal → organism`
- **Hyponym** — more specific concept: `dog → poodle, labrador, husky`

In [ ]:
def hypernym_chain(word: str, pos: str = wn.NOUN, depth: int = 8) -> List[str]:
    syns = wn.synsets(word, pos=pos)
    if not syns:
        return [word]
    syn   = syns[0]
    chain = [word]
    for _ in range(depth):
        parents = syn.hypernyms()
        if not parents:
            break
        syn = parents[0]
        chain.append(syn.lemmas()[0].name())
    return chain


print("Hypernym chains (specific -> general):")
for word in ["dog", "car", "apple", "scientist", "python"]:
    chain = hypernym_chain(word)
    print(f"  {word}: {' -> '.join(chain)}")

print()
dog_syn   = wn.synsets("dog", pos=wn.NOUN)[0]
hyponyms  = sorted(h.lemmas()[0].name() for h in dog_syn.hyponyms())
print(f"Hyponyms of 'dog' (first 12): {hyponyms[:12]}")

### 9.4 Semantic Similarity

| Metric | Range | Description |
|--------|-------|-------------|
| `path_similarity` | 0–1 | Inverse of shortest path in taxonomy |
| `wup_similarity` | 0–1 | Wu-Palmer: uses lowest common subsumer depth |
| `lch_similarity` | 0–∞ | Leacock-Chodorow: path + depth |

Higher score = more semantically similar.

In [ ]:
def word_sim(w1: str, w2: str, pos: str = wn.NOUN) -> Dict[str, float]:
    s1 = wn.synsets(w1, pos=pos)
    s2 = wn.synsets(w2, pos=pos)
    if not s1 or not s2:
        return {}
    return {
        "path": s1[0].path_similarity(s2[0]),
        "wup":  s1[0].wup_similarity(s2[0]),
        "lch":  s1[0].lch_similarity(s2[0]),
    }


pairs = [
    ("dog", "cat"),       # both mammals
    ("dog", "wolf"),      # closely related
    ("car", "automobile"),# synonyms
    ("car", "truck"),     # same category
    ("car", "apple"),     # unrelated
    ("man", "woman"),     # same level in hierarchy
]

print(f"{'Word 1':<12} {'Word 2':<12} {'path':<10} {'wup':<10} {'lch'}")
print("-" * 55)
for w1, w2 in pairs:
    sc = word_sim(w1, w2)
    if sc:
        path = f"{sc['path']:.3f}" if sc['path'] else "None"
        wup  = f"{sc['wup']:.3f}"  if sc['wup']  else "None"
        lch  = f"{sc['lch']:.3f}"  if sc['lch']  else "None"
        print(f"{w1:<12} {w2:<12} {path:<10} {wup:<10} {lch}")

print()
print("'car' and 'automobile' score near 1.0 (synonyms).")
print("'car' and 'apple' score near 0.0 (unrelated concepts).")

## 10. Frequency Distributions and Collocations

In [ ]:
from nltk import FreqDist
from nltk.corpus import gutenberg

print("Gutenberg corpus files:")
print(gutenberg.fileids())
print()

# Load Moby Dick
moby = [w.lower() for w in gutenberg.words("melville-moby_dick.txt") if w.isalpha()]
print(f"Moby Dick: {len(moby):,} alphabetic tokens")

fd = FreqDist(moby)
print(f"Vocabulary: {len(fd):,} unique words")
print()
print("Most common 15 words:")
for word, count in fd.most_common(15):
    bar = "#" * (count // 600)
    print(f"  {word:<12} {count:>6}  {bar}")

print()
for w in ["whale", "sea", "ahab", "ship", "captain"]:
    print(f"  '{w}': {fd[w]} occurrences")

In [ ]:
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

# Filter stopwords before collocation mining
filtered_moby = [w for w in moby if w not in en_stops and len(w) > 2]

finder = BigramCollocationFinder.from_words(filtered_moby)
finder.apply_freq_filter(5)   # ignore pairs appearing fewer than 5 times

print("Top 15 bigram collocations in Moby Dick (PMI score):")
print("=" * 50)
for bigram, score in finder.score_ngrams(BigramAssocMeasures.pmi)[:15]:
    print(f"  {str(bigram):<32} PMI = {score:.2f}")

print()
print("PMI ranks bigrams that co-occur more than chance.")
print("High PMI = words that appear together very specifically.")

## 11. NLTK vs Custom `text_preprocessing.py`

A direct side-by-side demonstration on the same corpus.

In [ ]:
from src.utils.text_preprocessing import (
    normalize_text, basic_tokenize, build_vocab, texts_to_sequences
)

corpus = [
    "The researchers are studying advanced language models carefully.",
    "Better models don't always produce better results for every task.",
    "Dr. Smith's team ran three experiments at the U.S. university.",
]

print("APPROACH 1: Custom text_preprocessing.py  (DL pipeline)")
print("=" * 65)
for text in corpus:
    cleaned = normalize_text(text, lowercase=True, expand_contractions=True)
    tokens  = basic_tokenize(cleaned)
    print(f"  IN:      {text}")
    print(f"  Cleaned: {cleaned}")
    print(f"  Tokens:  {tokens}")
    print()

vocab = build_vocab(corpus, max_vocab_size=30)
seqs  = texts_to_sequences(corpus, vocab, max_len=10)
print(f"Vocab size: {len(vocab)}")
print("Padded integer sequences:")
for s in seqs:
    print(f"  {s}")

In [ ]:
print("APPROACH 2: NLTK  (linguistic accuracy)")
print("=" * 65)
for text in corpus:
    tokens     = word_tokenize(text.lower())
    tagged     = pos_tag(tokens)
    lemmas     = [lemmatizer.lemmatize(tok, pos=treebank_to_wn(tag)) for tok, tag in tagged]
    meaningful = [l for l in lemmas if l not in en_stops and l.isalpha()]
    print(f"  IN:          {text}")
    print(f"  Tokens:      {tokens}")
    print(f"  Lemmas:      {lemmas}")
    print(f"  Meaningful:  {meaningful}")
    print()

print("-" * 65)
print("Observations:")
print("  basic_tokenize : don't -> ['don', """'""", 't']   (regex)")
print("  word_tokenize  : don't -> ['do', \"n't\"]          (Treebank)")
print("  custom         : 'studying' stays 'studying'")
print("  NLTK lemma     : 'studying' -> 'study', 'ran' -> 'run'")

In [ ]:
# Full capability matrix
capabilities = [
    ("URL / email removal",         "YES (regex)",       "No (manual)"),
    ("Contraction expansion",        "YES (9 rules)",     "No (manual)"),
    ("Emoji removal",                "YES",               "No"),
    ("Word tokenization",            "Regex (naive)",     "Treebank (trained)"),
    ("Sentence tokenization",        "No",                "Punkt (trained)"),
    ("Tweet tokenization",           "No",                "TweetTokenizer"),
    ("Stopwords (25 languages)",     "No",                "YES"),
    ("Stemming",                     "No",                "Porter/Snowball/Lancaster"),
    ("Lemmatization",                "No",                "WordNet (POS-aware)"),
    ("POS tagging",                  "No",                "Perceptron tagger"),
    ("Named entity recognition",     "No",                "MaxEnt chunker"),
    ("WordNet / semantic similarity","No",                "YES (155k words)"),
    ("Frequency distributions",      "Counter only",      "FreqDist + collocations"),
    ("Vocab -> integer sequences",   "YES",               "No"),
    ("Batch padding for DL",         "YES",               "No"),
    ("Zero dependencies",            "YES (stdlib only)", "No (nltk + data)"),
    ("Speed",                        "Very fast",         "Moderate"),
]

print(f"{'Capability':<35} {'text_preprocessing.py':<25} {'NLTK'}")
print("-" * 80)
for cap, custom, nltk_col in capabilities:
    print(f"{cap:<35} {custom:<25} {nltk_col}")

## 12. Building an NLTK Pipeline

A reusable pipeline that applies NLTK preprocessing and feeds results into the DL vocab builder.

In [ ]:
class NLTKPipeline:
    """Tokenize -> POS-tag -> Lemmatize -> filter stopwords."""

    def __init__(
        self,
        remove_stops:  bool = True,
        alpha_only:    bool = True,
        min_len:       int  = 2,
        keep_negation: bool = False,
    ):
        self.stop_set = set(stopwords.words("english"))
        if keep_negation:
            self.stop_set -= {"no", "not", "nor", "neither", "never"}
        self.remove_stops = remove_stops
        self.alpha_only   = alpha_only
        self.min_len      = min_len

    def _wn_pos(self, tag: str) -> str:
        if tag.startswith("J"): return wn.ADJ
        if tag.startswith("V"): return wn.VERB
        if tag.startswith("N"): return wn.NOUN
        if tag.startswith("R"): return wn.ADV
        return wn.NOUN

    def process(self, text: str) -> List[str]:
        tokens = word_tokenize(text.lower())
        tagged = pos_tag(tokens)
        result = [lemmatizer.lemmatize(tok, pos=self._wn_pos(tag)) for tok, tag in tagged]
        if self.alpha_only:   result = [t for t in result if t.isalpha()]
        if self.remove_stops: result = [t for t in result if t not in self.stop_set]
        result = [t for t in result if len(t) >= self.min_len]
        return result

    def process_batch(self, texts: List[str]) -> List[List[str]]:
        return [self.process(t) for t in texts]

    def to_string(self, text: str) -> str:
        return " ".join(self.process(text))


# Compare two configs
bow_pipe       = NLTKPipeline(remove_stops=True,  keep_negation=False)
sentiment_pipe = NLTKPipeline(remove_stops=True,  keep_negation=True)

samples = [
    "The researchers were studying advanced language models at MIT.",
    "This product is not good at all -- I would not recommend it.",
    "Dr. Smith's experiments ran successfully over three days.",
]

print("Pipeline output:")
print("=" * 70)
for text in samples:
    print(f"  Input:          {text}")
    print(f"  BoW pipeline:   {bow_pipe.process(text)}")
    print(f"  Sentiment pipe: {sentiment_pipe.process(text)}")
    print()

In [ ]:
# Combine with custom DL utilities
dl_corpus = [
    "The cat sat on the mat and stared at the dog.",
    "Dogs are running faster than cats in the park.",
    "Better models produce better results for natural language processing.",
    "The researcher studied the behavior of mice and geese.",
]

pipe = NLTKPipeline(remove_stops=True, min_len=3)
processed = [pipe.to_string(t) for t in dl_corpus]

print("NLTK preprocessed -> custom DL vocab + sequences:")
print("=" * 60)
for orig, proc in zip(dl_corpus, processed):
    print(f"  IN:  {orig}")
    print(f"  OUT: {proc}")
    print()

vocab = build_vocab(processed, max_vocab_size=40)
seqs  = texts_to_sequences(processed, vocab, max_len=8)
print(f"Vocab size: {len(vocab)}")
print("Padded sequences (max_len=8):")
for s in seqs:
    print(f"  {s}")

## 13. Common Mistakes and Debugging Tips

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Missing `punkt` / `punkt_tab` data | `LookupError` on `word_tokenize` | `nltk.download('punkt punkt_tab')` |
| Calling `lemmatize(w)` without POS | Default noun — `"running"` stays `"running"` | Pass `pos='v'` for verbs |
| POS-tagging after lowercasing | `"Apple"` loses proper-noun signal | POS-tag original case, lowercase after |
| Removing all stopwords for sentiment | `"not good"` → `"good"` (meaning reversed) | Set `keep_negation=True` |
| Stemming proper nouns | `"London"` → `"london"` / `"lond"` | Lemmatize or exclude NNP tokens |
| WordNet similarity across POS | Returns `None` — noun vs verb hierarchy separate | Ensure both synsets share the same POS |
| Collocation mining with stopwords | `("the", "a")` dominates all bigrams | Filter stopwords before `BigramCollocationFinder` |
| No `apply_freq_filter` on collocations | Rare pairs inflate PMI artificially | Always set `apply_freq_filter(n >= 5)` |
| Using NLTK tokenizer before subword model | BERT expects raw text, not pre-tokenized | Feed raw text directly to subword tokenizer |

**Debugging tip:** `nltk.download('all')` fetches everything (~3.5 GB). For selective installs, list only what you need (see Section 2).

## 14. Exercises

### Exercise 1: Full NLTK Analysis Pipeline

Given `exercise_text`:
1. Split into sentences with `sent_tokenize`.
2. For each sentence, tokenize and POS-tag.
3. Extract all named entities.
4. Collect every **verb** token, lemmatize it, and print `(original, lemma)` for forms that changed.

In [ ]:
exercise_text = (
    "Elon Musk founded SpaceX in 2002 in Hawthorne, California. "
    "The company has been developing rockets and spacecraft. "
    "NASA awarded SpaceX a $2.9 billion contract for lunar landers. "
    "Tesla, another company he leads, produces electric vehicles "
    "and energy storage systems."
)

# TODO: Your analysis here
# 1. sentences = sent_tokenize(...)
# 2. for each sentence: tokens = word_tokenize(...), tagged = pos_tag(...)
# 3. entities via ne_chunk
# 4. verb lemmatization

### Exercise 2: WordNet Query Expansion

Implement `expand_query(query)` that:
1. Tokenizes and POS-tags the query.
2. For each noun and verb, fetches all WordNet synonyms.
3. Returns the union of original words + synonyms (lowercase, underscores → spaces).

In [ ]:
# TODO: implement expand_query
# def expand_query(query: str) -> set:
#     ...

# Test:
# for q in ["fast car", "happy dog", "student study"]:
#     expanded = expand_query(q)
#     print(f"Query '{q}': +{len(expanded - set(q.split()))} synonyms")
#     print(sorted(expanded)[:10])

### Exercise 3: Compare Preprocessing Strategies

Using `exercise_corpus` below, build and evaluate three TF-IDF + Logistic Regression classifiers:
1. Raw lowercased text
2. Porter stemmed (no stopwords)
3. NLTK lemmatized (stopwords removed)

Report 4-fold cross-validation accuracy for each.

In [ ]:
import numpy as np

exercise_corpus = [
    # Tech (0)
    "The engineers designed a new machine learning algorithm.",
    "Software developers are debugging the production server.",
    "The neural network achieved 98% accuracy on the test set.",
    "Open-source frameworks are driving innovation in AI research.",
    "Cloud computing enables scalable data storage solutions.",
    "The API endpoint returns JSON responses to client applications.",
    # Sports (1)
    "The team scored three goals in the final quarter of the match.",
    "Athletes train rigorously to improve their performance.",
    "The marathon runner finished in under two hours and set a record.",
    "Football clubs are competing fiercely for the championship trophy.",
    "The swimmer broke the world record in the freestyle event.",
    "Coaches are analyzing game strategies to win the next tournament.",
]
exercise_labels = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])

# TODO: build and compare three classifiers

## 15. Exercise Solutions

In [ ]:
# ---- Exercise 1 Solution ----
print("Exercise 1: Full NLTK Analysis")
print("=" * 60)

sentences = sent_tokenize(exercise_text)
print(f"{len(sentences)} sentences detected:")
for i, s in enumerate(sentences, 1):
    print(f"  [{i}] {s}")
print()

all_entities   = []
all_verb_pairs = []

for sent in sentences:
    tokens  = word_tokenize(sent)
    tagged  = pos_tag(tokens)
    chunked = ne_chunk(tagged)

    for subtree in chunked:
        if isinstance(subtree, Tree):
            span = " ".join(w for w, t in subtree.leaves())
            all_entities.append((span, subtree.label()))

    for word, tag in tagged:
        if tag.startswith("V"):
            lemma = lemmatizer.lemmatize(word.lower(), pos=wn.VERB)
            if word.lower() != lemma:
                all_verb_pairs.append((word, lemma))

print("Named Entities:")
for ent, etype in all_entities:
    print(f"  [{etype}] {ent}")

print()
print("Verb Lemmatization (changed forms):")
for orig, lemma in all_verb_pairs:
    print(f"  {orig:<15} -> {lemma}")

In [ ]:
# ---- Exercise 2 Solution ----
def expand_query(query: str) -> set:
    tokens  = word_tokenize(query.lower())
    tagged  = pos_tag(tokens)
    result  = set(tokens)
    for word, tag in tagged:
        if tag.startswith("N"):   pos = wn.NOUN
        elif tag.startswith("V"): pos = wn.VERB
        elif tag.startswith("J"): pos = wn.ADJ
        else: continue
        for syn in wn.synsets(word, pos=pos):
            for lemma in syn.lemmas():
                result.add(lemma.name().lower().replace("_", " "))
    return result


print("Exercise 2: WordNet Query Expansion")
print("=" * 60)
queries = ["fast car", "happy dog", "student study", "run quickly"]
for q in queries:
    original = set(q.lower().split())
    expanded = expand_query(q)
    added    = expanded - original
    print(f"  Query: '{q}'")
    print(f"  Added synonyms ({len(added)}): {sorted(added)[:8]}")
    print()

In [ ]:
# ---- Exercise 3 Solution ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

nltk_pl = NLTKPipeline(remove_stops=True, min_len=3)

raw_texts    = [t.lower() for t in exercise_corpus]
stemmed_texts = [
    " ".join(porter.stem(w) for w in word_tokenize(t.lower())
              if w.isalpha() and w not in en_stops)
    for t in exercise_corpus
]
lemma_texts  = [nltk_pl.to_string(t) for t in exercise_corpus]

configs = [
    ("Raw lowercase",            raw_texts),
    ("Porter stem + no stops",   stemmed_texts),
    ("NLTK lemma + no stops",    lemma_texts),
]

print("Exercise 3: Classification Accuracy Comparison")
print("=" * 55)
for name, texts_list in configs:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf",   LogisticRegression(random_state=42, max_iter=1000)),
    ])
    scores = cross_val_score(pipe, texts_list, exercise_labels, cv=4, scoring="accuracy")
    print(f"  {name:<28}  {scores.mean():.3f} (+/- {scores.std():.3f})")

print()
print("Sample output after each strategy (first text):")
for name, texts_list in configs:
    print(f"  {name}: {texts_list[0]}")